# 🔗 시스템 통합 종합 정리 노트 (6장 — 모듈 통합 · 최적화 · 서버 구축 · 배포 · 테스트)

> **생성형 AI 기반 음성 에이전트 개발 과정 · 6장 시스템 통합 파트 리뷰**
> `Modules/` 폴더 실습 노트북 중 **6장 5종**(6-1 통합 설계 · 6-2 파이프라인 최적화 · 6-3 FastAPI·WebRTC API · 6-4 GCP/Colab · 6-5 통합 테스트)을
> 하나로 정리한 **복습·재사용용 노트**입니다. (7장 프로젝트는 `08-project-design.ipynb`에서 별도 정리)

| 항목 | 내용 |
|---|---|
| 대상 노트북 | `Modules/` 6-1 ~ 6-5 (6-3 서버 구축 포함) |
| 노트의 목적 | ① **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | **macOS (Apple Silicon)** — 계약·오케스트레이터·계측·테스트는 전부 로컬 실행 |
| 관계 노트 | **1권 LLM** 사고부 계약(INTENT_SCHEMA) · **TTS/Cloning** 계약 · **7장** 프로젝트 |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **GPU·모델·키·네트워크 없이** 실행되는 계약·어댑터·오케스트레이터·
> 지연 계측·테스트·전송 계약(wire)만 모았습니다. 실제 서버(uvicorn/websocket) 기동·WebRTC·GCP 배포는
> 시그니처+가이드로 요약했습니다 (📄). 위에서 아래로 실행하세요.


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 (6장 5종 · macOS 실행 판정) |
| **1** | 실험에 필요한 선행 지식 (계약 3종 · 어댑터 ABC · 예산 게이트 · 서버 전송·wire 계약) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (파이프라인 최적화 · 서버 기동 · 배포 · 테스트 + macOS 가이드) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 6장 5종 한눈에

| 노트북 | 주제 | **macOS 판정** | 코멘트 |
|---|---|---|---|
| **6-1** | 모듈 통합 설계 — 계약·어댑터·오케스트레이터 | ✅ 전부 실행 | Base ABC + Mock 엔진 + VoiceAgentPipeline |
| **6-2** | 파이프라인 최적화 — 지연 프로파일·reprompt·캐시·오버랩 | ✅ 전부 실행 | pctl·CachedTTS·run_overlap·playback_timeline |
| **6-3** | FastAPI·WebRTC API 구축 — REST v1 · WS v2/v3 · WebRTC | ⚠️ 전송 계약만 실행 / 서버는 📄+3장 가이드 | agent_v1/v2/v3·프레임 뭉침·to_thread·SDP |
| **6-4** | GCP/Colab 실행 환경 — 노트북 탈출·터널링·배포 | ⚠️ 계약 로직만 / 서버는 로컬 기동 | 바인딩 매트릭스·cloudflared·배포 5종 |
| **6-5** | 시스템 통합 테스트 — pytest 스위트·장애 주입 | ⚠️ 계약 로직만 / pytest 로컬 | live_server·x-fault·종료코드 판정 |

> **핵심 흐름**: 6-1(부품을 계약으로 꽂는다) → 6-2(병목을 계측으로 찾고 고친다) → 6-3(그걸 서버로 띄운다) → 6-4(서버를 노트북 밖으로 내보낸다) → 6-5(전체를 자동으로 증명한다).

## 0-2. 계약·어댑터·게이트 구조 (6-1의 뼈대)

```
[ASR 엔진] → BaseASR.transcribe → make_asr_record + validate_asr_contract
[LLM 엔진] → BaseLLM.generate   → INTENT_SCHEMA (validate_llm_result)
[TTS 엔진] → BaseTTS.synth      → make_tts_record + validate_tts_contract
        ↑ VoiceAgentPipeline.run (오케스트레이터 — 예산·오류정책·트레이스)
        ↓ (6-3) 서버 포장 — 계약이 프로세스 경계를 넘는다
        [REST v1: 한 방] · [WS v2/v3: 메타+바이너리 프레임 쌍 + end] · [WebRTC: UDP 실시간 미디어]
```


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전 훑기 → ② 1장 선행 지식 → ③ 2장 함수 실행하며
> "검증 통과 ✅" 눈으로 확인. 모르는 단어는 여기로 돌아오세요.

### A. 계약과 어댑터 — "부품을 규격으로 꽂는다"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 계약 (contract) | 모듈이 지켜야 하는 최소 형식(레코드 키) | 엔진이 바뀌어도 뒤 코드는 그대로 |
| ASR 계약 | utt_id·text·language·confidence·engine·elapsed_ms | 듣기 결과의 표준 |
| TTS 계약 | audio·sr·num_channels·dtype·text·duration_s | 말하기 결과의 표준 (16k mono f32) |
| LLM 계약 | INTENT_SCHEMA (enum·required) | 생각 결과의 표준 |
| 어댑터 (adapter) | 엔진의 '자기 모양'을 계약으로 접는 벽 | (segments, info) 튜플 등 흡수 |
| ABC | 추상 기본 클래스 (BaseASR 등) | "무엇이든 꽂히게" 하는 인터페이스 |
| 확장은 자유 | 필수 키 외 추가 키(emotion 등) 허용 | 계약은 최소 보장 |

### B. 오케스트레이터 — "3단계를 예산·오류정책으로 조립"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 오케스트레이터 | ASR→LLM→TTS 순서를 조립·검증 | VoiceAgentPipeline.run |
| 예산 (budget) | 단계별·총합 시간 한도 | BUDGET_MS = {ASR:500, LLM:1500, TTS:800, TOTAL:3000} |
| 오류 정책 | fail_fast(전파) vs fallback(안전 응답) | 실패 시 행동 규칙 |
| 트레이스 (trace) | 단계별 엔진·시간·성공 여부 기록 | "무엇이 얼마나 걸렸나" 표 |
| fallback | 장애 시 안전 응답(상담사 연결) | 최악에도 응답은 나간다 |

### C. 파이프라인 최적화
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| p50 / p95 | 백분위 지연 — 95%는 이 안에 끝난다 | 꼬리 지연을 보는 기준선 |
| reprompt 게이트 | 신뢰도 낮으면 LLM 비용 안 쓰고 재발화 요청 | 불필요 호출 제거 |
| CachedTTS | 상용구(인사·폴백) 캐시 | 첫 합성 후 ~0ms |
| 오버랩 | 문장 도착 즉시 합성 (생산-소비) | TTFA 단축 |
| 언더런 (underrun) | 버퍼가 비어 재생 끊김 | 재생 타임라인으로 검출 |

### D. 서버·테스트
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| REST (agent_v1) | HTTP POST 오디오 → JSON 응답 | 간단한 요청-응답 |
| WebSocket (agent_v3) | 양방향·스트리밍 채널 | chunk 단위로 오디오 흘림 |
| wire 계약 | type(seq/text/dur_ms) 메타 검증 | 프레임 뭉침·손상 방지 |
| to_thread | 블로킹 작업을 스레드로 위임 | 이벤트 루프는 항상 자유 |
| 장애 주입 | x-fault: llm 헤더로 고장 엔진 교체 | "고장을 일부러 만들어" 검증 |
| 종료코드 | pytest 반환 값(0=성공) | 테스트의 진실 = 출력이 아니라 rc |
| 카오스 스위치 | 장애 주입은 기본 OFF | 정상 경로가 기본 |

### E. 배경 지식 — 이 챕터가 왜 존재하는가
ASR(귀)·LLM(뇌)·TTS(입)를 하나로 묶는 것이 **시스템 통합**입니다. 이 노트의 중심 사상:
1. **계약으로 이식성 확보** — 각 모듈이 자기 멋대로 뱉으면 뒤에서 매번 깨진다. 계약 레코드로 접고,
   어댑터가 "엔진 모양"을 흡수한다. 그래서 엔진 교체는 **설정 한 줄**이다.
2. **예산과 계측** — "얼마나 걸리나"를 p50이 아니라 p95로 봐야 꼬리 지연이 보인다. 병목은 직감이 아니라
   계측이 찾는다. (reprompt·캐시·오버랩이 그 답)
3. **증명은 종료코드** — 계약·파이프라인·E2E 테스트를 pytest로 자동화하고, 장애를 일부러 주입해
   폴백이 진짜 동작하는지 확인한다. "회귀는 회귀 테스트가 지킨다."

### F. 전송 계약과 서버 (6-3)
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 전송 계약 (wire) | 프로세스 경계를 넘는 최소 형식 — `float32→int16 PCM→base64` | numpy 배열은 JSON을 못 탄다 — 네트워크는 바이트만 나른다 |
| PCM16 | 16kHz mono PCM_16 표준과 정렬된 정수 오디오 | 3주차의 오디오 계약이 전선(wire) 위로 확장 |
| base64 팽창 | JSON에 실으면 4/3배 부풀어 오름 | WS 바이너리 프레임에서는 세금 없음 |
| REST 한 방 (v1) | POST 오디오 → 전체 응답 한 덩어리 | **구조적으로 TTFA = 총지연** (첫 오디오 = 마지막 바이트) |
| 종료 신호 (end) | 스트림 끝을 알리는 명시적 프레임 `{"type":"end"}` | 없으면 교착/비정상 close — 완료를 확정할 수 없다 |
| 프레임 뭉침 | async 핸들러 속 블로킹이 루프를 잠가 프레임이 한꺼번에 쏟아짐 | TTFA 역주행 + **다른 연결도 함께 정지** (동시성 파괴) |
| to_thread | 블로킹 호출을 워커 스레드로 위임 (`asyncio.to_thread`) | 루프는 항상 자유로워야 플러시·동시성이 산다 |
| WebRTC | UDP 계열(SRTP) 실시간 미디어 표준 | "늦은 패킷은 버리고 간다" — TCP의 재전송 대기(HoL) 회피 |
| SDP | 오퍼/앤서를 나르는 시그널링 문서 (`m=`·`ice-ufrag`·`fingerprint`) | 시그널링은 표준 밖 — 우리 몫, WS가 배달부가 된다 |
| DataChannel | WebRTC의 제어/메타 채널 (SCTP) | 6-3 데모에서 wire 메타를 에코해 RTT를 잰다 |


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 핵심 개념 |
|---|---|---|---|
| 2.0 | `validate_asr_contract`·`validate_tts_contract`·`validate_llm_result`·`validate_wire_meta` | 계약 3종 + wire | 최소 보장·위반 포착 |
| 2.1 | `BaseASR/BaseLLM/BaseTTS`·`Mock*`·`VoiceAgentPipeline` | 어댑터 + 오케스트레이터 | 계약·예산·트레이스 |
| 2.2 | `BrokenLLM`·조합 매트릭스 | 오류 정책 | fail_fast/fallback |
| 2.3 | `pctl`·`profile`·`VoiceAgentPipelineV2`·`CachedTTS` | 최적화 | p95·reprompt·캐시 |
| 2.4 | `run_serial`·`run_overlap`·`playback_timeline` | 직렬 vs 오버랩 | TTFA·언더런 |
| 2.5 | `test_*`·`pick_llm` | 통합 테스트 | 장애 주입·회귀 |
| 2.6 | `audio_to_wire`·`wire_to_audio` | 전송 계약 왕복 | PCM16 base64·위반 포착 |
| 📄 2.6보충 | `agent_v1/v2/v3`·`smoke_test_63`·DataChannel 루프백 | 서버 엔드포인트 3종 | 프레임 뭉침·to_thread·WebRTC |


# 1. 실험에 필요한 선행 지식 🧠

## 1-1. 계약 3종 — "엔진 모양을 어댑터가 계약으로 접는다"

| 계약 | 생성 | 검증 | 핵심 |
|---|---|---|---|
| ASR | `make_asr_record(utt_id, text, language, confidence, engine, elapsed_ms)` | `validate_asr_contract` | text 비어있지 않음 · 0≤confidence≤1 |
| TTS | `make_tts_record(audio, sr, text)` | `validate_tts_contract` | **16kHz mono float32** · 키 6종 |
| LLM | (어댑터 내부 파싱) | `validate_llm_result` | INTENT_SCHEMA (enum·required) |

**교훈 (6-1 셀 5)**: 엔진 A는 `(segments, info)` 튜플·dict·`(wav, sr)` 튜플을 돌려주는 자기만의 모양을 가진다.
어댑터는 그 **모양을 흡수해 계약 레코드로 접는다** — "무엇이든 꽂히게" 하는 벽.

## 1-2. 어댑터 ABC (BaseASR/BaseLLM/BaseTTS) — 개방-폐쇄의 최소 구현

```python
class BaseLLM(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> dict:
        # INTENT_SCHEMA를 통과하는 dict 반환 — 파싱·펜스 제거는 어댑터 내부 책임
```
- 등록된 어댑터는 **인터페이스 규약을 지켜야** 조합 매트릭스(2×2×1)에 들어간다.
- `BadMockTTS`(24kHz)는 어댑터가 리샘플을 게을리하면 계약 검증에서 **크게 실패** — "성공 선언은 게이트 통과 뒤에만".

## 1-3. 오케스트레이터 VoiceAgentPipeline — 예산 · 오류 정책 · 트레이스

```
run(): ① ASR(계약) → ② LLM(실패 시 fail_fast/fallback 분기) → ③ TTS(계약) → ④ 예산 판정
```
- **예산** `BUDGET_MS = {"ASR":500, "LLM":1500, "TTS":800, "TOTAL":3000}` — 단계별·총합 초과 판정.
- **오류 정책**: `fail_fast`(예외 전파) vs `fallback`(안전 응답 + 트레이스에 실패 기록).
- **트레이스** `make_trace(stage, engine, ms, ok, note)` — 어떤 엔진이 얼마나 걸렸는지 표로.

## 1-4. 파이프라인 최적화 (6-2) — 지연의 세 가지 수술

1. **계측 먼저**: `profile(pipe, n)` 30회 → `pctl(xs, q)` 백분위 — **p50 대신 p95를 봐야** 꼬리 지연이 보인다.
2. **reprompt 게이트**: ASR confidence < 0.6 → LLM 비용을 아예 쓰지 않고 재발화 요청 (`VoiceAgentPipelineV2`).
3. **캐시**: `CachedTTS` — 상용구(인사·폴백)는 첫 합성 후 ~0ms (FIFO 축출 64개, 과제: LRU).
4. **직렬 vs 오버랩**: 문장 도착 즉시 합성(생산자-소비자) → TTFA 단축. `playback_timeline`으로 **언더런(끊김)** 감지.

## 1-5. 서버 전송 (6-3) — REST → WS → WebRTC

- **전송 계약(wire)**: 계약 레코드의 numpy 배열은 JSON을 못 탄다 → `float32 → int16 PCM → base64` (과정 표준 16kHz mono PCM_16과 정렬, JSON엔 4/3 팽창 세금). 2.6에서 왕복 검증.
- **REST v1** (`/v1/agent`): POST 오디오 → 파이프라인 한 번 → JSON(오디오는 `audio_b64`). **구조적 한계**: 응답이 한 덩어리 → TTFA = 총지연 (첫 오디오 = 마지막 바이트).
- **WS v2** (`/v2/agent`): JSON 메타(`type: chunk|end`, `seq`, `text`, `dur_ms`) + **바이너리 PCM16** 프레임 쌍. `dur_ms` 덕에 클라이언트가 2.4의 언더런 판정기를 그대로 돌린다. **종료 신호 `{"type":"end"}`도 계약의 일부** — 잊으면 상호 대기 교착(`TimeoutError`) 또는 비정상 close(`ConnectionClosedError`)로 클라이언트가 완료를 확정할 수 없다.
- **프레임 뭉침 사고 (6-3의 핵심 교훈)**: `async def` 핸들러 안의 블로킹(`time.sleep`, 실물은 GPU 추론·외부 API)이 이벤트 루프를 잠그면, `await send_bytes()`는 전송 버퍼에 쓸 뿐 **플러시는 루프가 한가할 때** 일어난다 → 핸들러가 끝나는 순간 프레임이 한꺼번에 쏟아진다. 루프가 잠긴 동안 **이 서버의 다른 연결도 함께 정지** (동시성 파괴).
- **교정 v3** (`/v3/agent`): async 핸들러 속 **모든 블로킹을 `asyncio.to_thread`로 추방** — 블로킹 제너레이터의 소비(`next`)까지 포함. 루프는 항상 자유.
- **회귀 게이트**: `arrive[-1] - arrive[0] > 500` (뭉침 없음) · `WS_TTFA < REST_TTFA` · 언더런 0 · `TTFA_BUDGET_MS=1500`.
- **WebRTC**: WS는 TCP라 재전송 대기(head-of-line blocking)로 늦은 패킷이 뒤 유효 패킷까지 막는다. 전화 품질 음성은 **"늦은 패킷은 버리고 간다"** — UDP 계열(SRTP) + 지터 버퍼·AEC·대역폭 적응을 표준으로 내장. SDP 오퍼/앤서 시그널링은 표준 **밖**(우리 몫, 보통 WS가 배달부), DataChannel(SCTP)로 제어 메타를 나른다.

## 1-6. GCP/Colab 실행 환경 (6-4) — 서버를 노트북 밖으로

- **이 기계의 정체**: Colab은 GPU가 '노트북이 아니라 서버'에 붙은 원격 머신 — 세션은 오늘 밤 사라진다.
- **노트북 탈출**: 서버를 파일(server.py)로 추출 → **별도 프로세스**로 기동 (바인딩 매트릭스 실측: 내부 IP → loopback 불가 / 외부 가능). **포트는 자원** — 충돌 실증.
- **터널링**: cloudflared로 NAT 안에서 공인 인터넷으로 문 내기; 거리의 비용을 RTT·지터로 실측.
- **배포 산출물**: server.py · requirements.txt · Dockerfile · deploy_cloudrun.sh · deploy_gce_t4.sh.

## 1-7. 시스템 통합 테스트 (6-5) — 종료코드가 오늘의 주인공

- **pytest 별도 프로세스**: `run_pytest(*args, expect_rc=...)` — 실패는 출력이 아니라 **종료코드**로 판정.
- **장애 주입**: `x-fault: llm` 헤더 → `pick_llm`이 BrokenLLM으로 교체 — "카오스 스위치는 기본 OFF".
- **위반 포착**: `test_`로 시작 안 하면 pytest가 모른다(유령 테스트) — 명명 규약 자체가 게이트.
- **회귀 테스트**: 프레임 뭉침(WS) 사고를 영구 검증 — `arrive[-1] - arrive[0] > 500`.
- **V4**: `with pytest.raises(AssertionError)` — `_pytest/raises.py:74` 관례.
- **finally의 자리**: `live_server`가 테스트 실패해도 포트는 반납된다.

## 1-8. macOS 실행 가이드

| 항목 | 판정 | 설명 |
|---|---|---|
| 계약·어댑터·오케스트레이터 (6-1) | ✅ 전부 실행 | 이 노트 2.0~2.1 |
| 지연 계측·캐시·오버랩 (6-2) | ✅ 전부 실행 | 이 노트 2.3~2.4 |
| 전송 계약 wire 왕복 (6-3) | ✅ 전부 실행 | 이 노트 2.6 (numpy만) |
| 서버 기동 (6-3·6-4) | ⚠️ pip 필요 | `pip install fastapi uvicorn websockets httpx` — 로컬에서 실행 가능 (3-2 가이드) |
| WebRTC 루프백 (6-3) | ⚠️ pip 필요 | `pip install aiortc nest_asyncio` — macOS·Colab 공통 (3-2 가이드) |
| pytest 스위트 (6-5) | ⚠️ pip 필요 | `pip install pytest httpx` — 로컬에서 실행 가능 |
| GCP 배포 | ⚠️ 클라우드 | Cloud Run / GCE(T4) — 키·프로젝트 필요 |

> 계약·계측·게이트 로직은 **어떤 머신에서든 동일하게 실행** — 실측 수치(ms)만 환경 의존.


# 2. 함수/클래스 정의 및 주석 🔧

> ✅ = 실행 코드 셀 (GPU·모델·키 없이 assert 자가점검) · 📄 = 요약만 (실물 서버/배포는 3장 가이드)


In [ ]:
# ═══ 2.0 공통 계약 3종 + wire 계약 (6-1 · 6-4) ✅ ═══
# ▶ 계약 3종은 '엔진이 지켜야 할 최소 형식' — 위반은 즉시 AssertionError(시끄러운 실패).
#   TTS 계약의 핵심: 16kHz·모노·float32 — 24kHz/스테레오를 경계에서 차단한다.
import json, time, re, math, random, gc
import numpy as np
import jsonschema

INTENTS = ["billing", "tech_support", "loss_suspend", "plan_change", "payment_change"]
INTENT_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": INTENTS},
        "slots": {"type": "object"},
        "reply": {"type": "string", "minLength": 1},
        "handoff_to_human": {"type": "boolean"},
    },
    "required": ["intent", "slots", "reply", "handoff_to_human"],
    "additionalProperties": False,
}
ASR_CONTRACT_KEYS = ("utt_id", "text", "language", "confidence", "engine", "elapsed_ms")
TTS_CONTRACT_KEYS = ("audio", "sr", "num_channels", "dtype", "text", "duration_s")
WIRE_META_KEYS = ("type", "seq", "text", "dur_ms")

def make_asr_record(utt_id, text, language, confidence, engine, elapsed_ms):
    return {"utt_id": utt_id, "text": str(text), "language": str(language),
            "confidence": float(confidence), "engine": str(engine),
            "elapsed_ms": float(elapsed_ms)}

def validate_asr_contract(rec):
    missing = [k for k in ASR_CONTRACT_KEYS if k not in rec]
    assert not missing, "ASR 계약 위반 — 누락 키: " + str(missing)
    assert isinstance(rec["text"], str) and len(rec["text"]) > 0, "ASR 계약 위반 — text 비어 있음"
    assert 0.0 <= rec["confidence"] <= 1.0, "ASR 계약 위반 — confidence 범위: " + str(rec["confidence"])
    return True

def make_tts_record(audio, sr, text):
    audio = np.asarray(audio, dtype=np.float32)
    return {"audio": audio, "sr": int(sr), "num_channels": 1,
            "dtype": str(audio.dtype), "text": text,
            "duration_s": round(len(audio) / sr, 3)}

def validate_tts_contract(rec):
    missing = [k for k in TTS_CONTRACT_KEYS if k not in rec]
    assert not missing, "TTS 계약 위반 — 누락 키: " + str(missing)
    assert rec["sr"] == 16000, "표준 16kHz 위반: " + str(rec["sr"])
    assert rec["num_channels"] == 1, "모노 위반"
    assert rec["dtype"] == "float32", "dtype 위반: " + rec["dtype"]
    return True

def validate_llm_result(obj):
    jsonschema.validate(instance=obj, schema=INTENT_SCHEMA)
    return True

def validate_wire_meta(m):
    assert m.get("type") in ("chunk", "end", "reprompt"), "wire 계약 위반 — type: " + str(m.get("type"))
    if m["type"] == "chunk":
        missing = [k for k in WIRE_META_KEYS if k not in m]
        assert not missing, "wire 계약 위반 — 누락 키: " + str(missing)
    return True

# ── 위반-포착 (계약 도입과 동시에) ──
good_asr = make_asr_record("utt_001", "안녕하세요", "ko", 0.97, "mock-asr-a", 180.0)
assert validate_asr_contract(good_asr) is True
try:
    validate_asr_contract(make_asr_record("utt_001", "", "ko", 0.97, "e", 1.0))
    raise AssertionError("ASR 빈 text 미포착")
except AssertionError:
    pass
good_tts = make_tts_record(np.zeros(16000, np.float32), 16000, "정상 문장")
assert validate_tts_contract(good_tts) is True
try:
    validate_tts_contract(make_tts_record(np.zeros(12000, np.float32), 24000, "x"))
    raise AssertionError("TTS 24kHz 미포착")
except AssertionError:
    pass
assert validate_wire_meta({"type": "chunk", "seq": 0, "text": "x", "dur_ms": 1.0}) is True
assert validate_wire_meta({"type": "end"}) is True
try:
    validate_wire_meta({"type": "chunk", "seq": 0})     # 누락 키
    raise AssertionError("wire 누락 미포착")
except AssertionError:
    pass
print("공통 계약 검증 통과 ✅ — ASR·TTS(16k mono f32)·LLM(enum)·wire(type/seq/text/dur_ms)")


In [ ]:
# ═══ 2.1 어댑터 ABC + 오케스트레이터 VoiceAgentPipeline (6-1) ✅ ═══
# ▶ BaseABC(추상 인터페이스) + Mock 엔진(계약 지키는 예) + VoiceAgentPipeline(조립).
#   오케스트레이터는 엔진이 누구든 상관없다 — 계약만 지키면 된다 (개방-폐쇄).
from abc import ABC, abstractmethod

UTTS = {
    "utt_001": "안녕하세요, 지난달 요금이 평소보다 많이 나온 것 같아서 확인 부탁드려요.",
    "utt_002": "인터넷이 어제 저녁부터 자꾸 끊기는데 기사님 방문 예약할 수 있을까요?",
    "utt_003": "휴대폰을 분실해서 일단 정지하고 싶어요.",
    "utt_004": "결합 할인으로 바꾸면 한 달에 얼마나 저렴해지는지 알려주세요.",
    "utt_005": "자동이체 계좌를 다른 은행으로 변경하고 싶습니다.",
}
BUDGET_MS = {"ASR": 500, "LLM": 1500, "TTS": 800, "TOTAL": 3000}
FALLBACK_REPLY = {"intent": "billing", "slots": {},
                  "reply": "죄송합니다. 상담사를 연결해 드리겠습니다.",
                  "handoff_to_human": True}

def build_prompt(utt):
    return ("다음 고객 발화를 분석해 JSON으로만 응답하세요. 다른 텍스트나 코드블록 금지.\n"
            + "스키마: {\"intent\": \"billing|tech_support|loss_suspend|plan_change|payment_change\", "
            + "\"slots\": {}, \"reply\": \"고객 응대 문장(존댓말)\", \"handoff_to_human\": true|false}\n"
            + "고객 발화: " + utt)

def strip_fences(raw):
    s = raw.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s
        if s.endswith("```"):
            s = s[:-3]
    return s.strip()

def make_trace(stage, engine, ms, ok, note=""):
    return {"stage": stage, "engine": engine, "ms": round(float(ms), 1),
            "ok": bool(ok), "note": note}

class BaseASR(ABC):
    name = "base"
    @abstractmethod
    def transcribe(self, audio, utt_id: str) -> dict:
        pass   # make_asr_record 형식 반환 — 계약 통과 보장은 어댑터 책임

class BaseLLM(ABC):
    name = "base"
    @abstractmethod
    def generate(self, prompt: str) -> dict:
        pass   # INTENT_SCHEMA 통과 dict — 파싱·펜스 제거는 어댑터 내부

class BaseTTS(ABC):
    name = "base"
    @abstractmethod
    def synth(self, text: str) -> dict:
        pass   # validate_tts_contract 통과 레코드 (16kHz mono float32)

class MockASR(BaseASR):
    name = "mock-asr-a"
    def __init__(self, base_ms=180.0, jitter_ms=40.0):
        self.base_ms, self.jitter_ms = base_ms, jitter_ms
    def transcribe(self, audio, utt_id):
        t0 = time.perf_counter()
        segs = [{"text": UTTS[utt_id]}]
        info = {"language": "ko", "language_probability": 0.97}
        text = "".join(s["text"] for s in segs)
        time.sleep((self.base_ms + random.uniform(0, self.jitter_ms)) / 1000)
        ms = (time.perf_counter() - t0) * 1000
        return make_asr_record(utt_id, text, info["language"],
                               info["language_probability"], self.name, ms)

class MockLLM(BaseLLM):
    name = "mock-llm-a"
    _ROUTES = [("요금", "billing", "지난달 요금 내역을 확인해 안내드리겠습니다.", False),
               ("끊기", "tech_support", "기사 방문 예약을 도와드리겠습니다.", False),
               ("분실", "loss_suspend", "본인 확인 후 회선 정지를 진행하겠습니다. 지금 정지할까요?", True),
               ("결합", "plan_change", "결합 할인 적용 시 예상 요금을 안내드리겠습니다.", False),
               ("자동이체", "payment_change", "자동이체 계좌 변경을 도와드리겠습니다.", False)]
    def generate(self, prompt):
        time.sleep(random.uniform(0.25, 0.45))
        for kw, intent, reply, handoff in self._ROUTES:
            if kw in prompt:
                obj = {"intent": intent, "slots": {}, "reply": reply, "handoff_to_human": handoff}
                raw = "```json\n" + json.dumps(obj, ensure_ascii=False) + "\n```"
                parsed = json.loads(strip_fences(raw))
                validate_llm_result(parsed)
                return parsed
        raise ValueError("라우팅 실패: 알 수 없는 발화")

class MockTTS(BaseTTS):
    name = "mock-tts-a"
    def synth(self, text):
        time.sleep(random.uniform(0.10, 0.20))
        sr = 16000
        dur = max(0.5, 0.055 * len(text))
        t = np.linspace(0, dur, int(sr * dur), endpoint=False)
        wav = (0.1 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)
        rec = make_tts_record(wav, sr, text)
        validate_tts_contract(rec)          # 성공 선언은 게이트 통과 뒤에만
        return rec

class VoiceAgentPipeline:
    # ASR → LLM → TTS 오케스트레이터. 엔진은 계약만 지키면 무엇이든 꽂힌다
    def __init__(self, asr, llm, tts, budgets=BUDGET_MS, error_policy="fallback"):
        assert error_policy in ("fail_fast", "fallback")
        self.asr, self.llm, self.tts = asr, llm, tts
        self.budgets, self.error_policy = budgets, error_policy
    def _timed(self, fn, *args):
        t0 = time.perf_counter()
        out = fn(*args)
        return out, (time.perf_counter() - t0) * 1000
    def run(self, audio, utt_id):
        traces = []
        asr_rec, ms = self._timed(self.asr.transcribe, audio, utt_id)
        validate_asr_contract(asr_rec)
        traces.append(make_trace("ASR", self.asr.name, ms, True))
        try:
            llm_rec, ms = self._timed(self.llm.generate, build_prompt(asr_rec["text"]))
            validate_llm_result(llm_rec)
            traces.append(make_trace("LLM", self.llm.name, ms, True))
        except Exception as e:
            if self.error_policy == "fail_fast":
                raise
            llm_rec = dict(FALLBACK_REPLY)
            traces.append(make_trace("LLM", self.llm.name, 0.0, False, "폴백: " + type(e).__name__))
        tts_rec, ms = self._timed(self.tts.synth, llm_rec["reply"])
        validate_tts_contract(tts_rec)
        traces.append(make_trace("TTS", self.tts.name, ms, True))
        verdicts, total = {}, 0.0
        for t in traces:
            total += t["ms"]
            b = self.budgets.get(t["stage"])
            verdicts[t["stage"]] = ("통과" if t["ms"] <= b else "초과") if b else "-"
        verdicts["TOTAL"] = "통과" if total <= self.budgets["TOTAL"] else "초과"
        return {"utt_id": utt_id, "asr": asr_rec, "llm": llm_rec, "tts": tts_rec,
                "traces": traces, "verdicts": verdicts, "total_ms": round(total, 1)}

random.seed(42)
r = VoiceAgentPipeline(MockASR(), MockLLM(), MockTTS()).run({"pcm": None}, "utt_005")
assert r["llm"]["intent"] == "payment_change"
assert [t["stage"] for t in r["traces"]] == ["ASR", "LLM", "TTS"]
assert r["verdicts"]["TOTAL"] in ("통과", "초과")
print("오케스트레이터 검증 통과 ✅ — utt_005 완주 / 트레이스 3단계 / 예산 판정")


In [ ]:
# ═══ 2.2 오류 정책 + 조합 매트릭스 + 폴백 (6-1) ✅ ═══
# ▶ 오류 정책: fallback(안전 응답+트레이스 기록) vs fail_fast(예외 전파).
#   조합 매트릭스 2×2×1: 모든 어댑터 조합이 완주하는지 — '무엇이든 꽂히는' 증명.
class BrokenLLM(BaseLLM):
    name = "broken-llm"
    def generate(self, prompt):
        raise TimeoutError("모의 장애: 응답 시간 초과")

class MockASR_B(MockASR):
    # 지연 프로파일이 다른 두 번째 엔진 (더 무겁고 확신도 낮음)
    name = "mock-asr-b"
    def __init__(self):
        super().__init__(base_ms=340.0, jitter_ms=80.0)
    def transcribe(self, audio, utt_id):
        rec = super().transcribe(audio, utt_id)
        rec["engine"] = self.name
        rec["confidence"] = 0.88
        return rec

class MockLLM_B(MockLLM):
    # 펜스 없이 맨 JSON을 뱉는 엔진 — strip_fences는 무해해야 한다
    name = "mock-llm-b"
    def generate(self, prompt):
        for kw, intent, reply, handoff in self._ROUTES:
            if kw in prompt:
                time.sleep(random.uniform(0.15, 0.30))
                raw = json.dumps({"intent": intent, "slots": {}, "reply": reply,
                                  "handoff_to_human": handoff}, ensure_ascii=False)
                parsed = json.loads(strip_fences(raw))
                validate_llm_result(parsed)
                return parsed
        raise ValueError("라우팅 실패")

random.seed(1)
# ① fallback: LLM 장애 → 안전 응답 + 트레이스 실패 기록
r_fb = VoiceAgentPipeline(MockASR(), BrokenLLM(), MockTTS(), error_policy="fallback").run({"pcm": None}, "utt_001")
assert r_fb["llm"]["handoff_to_human"] is True          # FALLBACK_REPLY
assert any((not t["ok"]) and t["stage"] == "LLM" for t in r_fb["traces"])

# ② fail_fast: 예외 전파
try:
    VoiceAgentPipeline(MockASR(), BrokenLLM(), MockTTS(), error_policy="fail_fast").run({"pcm": None}, "utt_001")
    raise AssertionError("fail_fast 미전파")
except TimeoutError:
    pass

# ③ 조합 매트릭스: 2×2×1 전 조합이 utt_005 완주 (펜스 유무 무해성 포함)
registry = {"asr": [MockASR(), MockASR_B()], "llm": [MockLLM(), MockLLM_B()], "tts": [MockTTS()]}
combo_ok = True
for a in registry["asr"]:
    for l in registry["llm"]:
        res = VoiceAgentPipeline(a, l, registry["tts"][0]).run({"pcm": None}, "utt_005")
        combo_ok &= (res["llm"]["intent"] == "payment_change")
assert combo_ok
print("오류 정책 검증 통과 ✅ — fallback 안전응답 / fail_fast 전파 / 2×2×1 조합 매트릭스")


In [ ]:
# ▶ 데모 — '예산 초과'가 어떻게 판정되는지 눈으로 확인 (초보자용)
# 오케스트레이터는 각 단계와 총합이 BUDGET_MS 안인지 트레이스로 판정한다.

import random
random.seed(5)

# ① 정상 범위의 엔진: 예산 안에서 통과
pipe_ok = VoiceAgentPipeline(MockASR(), MockLLM(), MockTTS())
r_ok = pipe_ok.run({"pcm": None}, "utt_001")
print("① 정상:", {k: v for k, v in r_ok["verdicts"].items()}, f"총 {r_ok['total_ms']:.0f}ms")

# ② 느린 LLM: LLM 단계 예산(1500ms) 초과 시연
class SlowLLM(MockLLM):
    def generate(self, prompt):
        time.sleep(1.6)   # LLM 예산 1500ms를 넘기는 모의
        return super().generate(prompt)

r_slow = VoiceAgentPipeline(MockASR(), SlowLLM(), MockTTS()).run({"pcm": None}, "utt_001")
print("② 느린 LLM:", r_slow["verdicts"], f"총 {r_slow['total_ms']:.0f}ms")

assert r_ok["verdicts"]["TOTAL"] == "통과"
assert r_slow["verdicts"]["LLM"] == "초과" or r_slow["verdicts"]["TOTAL"] == "초과"
print("데모 통과 ✅ — 예산은 '무엇이 느린가'를 단계별로 가르쳐 주는 계기판")


In [ ]:
# ═══ 2.3 지연 계측 + reprompt 게이트 + 캐시 (6-2) ✅ ═══
# ▶ p95 계측: '대부분의 호출은 얼마나 걸리는가' — 평균은 꼬리 지연을 가린다.
#   reprompt 게이트: confidence<0.6이면 LLM 비용을 아예 안 쓴다 — '이것도 최적화다'.
#   CachedTTS: 상용구는 첫 합성 후 ~0ms — FIFO 축출(과제: LRU).
REPROMPT_REPLY = {"intent": "billing", "slots": {},
                  "reply": "죄송합니다, 잘 못 들었습니다. 다시 한번 말씀해 주시겠어요?",
                  "handoff_to_human": False}

def pctl(xs, q):
    return float(np.percentile(np.asarray(xs), q))

def profile(pipe, utt_id, n=12):
    totals, stage_ms = [], {"ASR": [], "LLM": [], "TTS": []}
    for _ in range(n):
        res = pipe.run({"pcm": None}, utt_id)
        totals.append(res["total_ms"])
        for t in res["traces"]:
            stage_ms[t["stage"]].append(t["ms"])
    return totals, stage_ms

class MockASR_Noisy(BaseASR):
    # 저신뢰 + 오인식 텍스트를 함께 반환 — 잡음 환경(SNR 5dB급) 재현
    name = "mock-asr-noisy"
    def transcribe(self, audio, utt_id):
        time.sleep(0.12)
        return make_asr_record(utt_id, "지난달 오금이 평소보다 많이 나왔어요",  # 요금→오금 오인식
                               "ko", 0.52, self.name, 120.0)

class VoiceAgentPipelineV2(VoiceAgentPipeline):
    # confidence 게이트 내장 — 임계 미만이면 LLM 비용을 아예 쓰지 않는다
    def __init__(self, *args, conf_threshold=0.6, **kw):
        super().__init__(*args, **kw)
        self.conf_threshold = conf_threshold
    def run(self, audio, utt_id):
        t0 = time.perf_counter()
        asr_rec, ms = self._timed(self.asr.transcribe, audio, utt_id)
        validate_asr_contract(asr_rec)
        if asr_rec["confidence"] < self.conf_threshold:
            traces = [make_trace("ASR", self.asr.name, ms, True, "저신뢰 " + str(asr_rec["confidence"]))]
            llm_rec = dict(REPROMPT_REPLY)
            tts_rec, ms2 = self._timed(self.tts.synth, llm_rec["reply"])
            validate_tts_contract(tts_rec)
            traces.append(make_trace("TTS", self.tts.name, ms2, True, "reprompt"))
            total = (time.perf_counter() - t0) * 1000
            return {"utt_id": utt_id, "asr": asr_rec, "llm": llm_rec, "tts": tts_rec,
                    "traces": traces, "verdicts": {"TOTAL": "통과"}, "total_ms": round(total, 1),
                    "reprompt": True}
        res = super().run(audio, utt_id)
        res["reprompt"] = False
        return res

class CachedTTS(BaseTTS):
    # 텍스트 키 캐시 — AICC 상용구(인사·폴백·reprompt)는 첫 합성 후 ~0ms
    name = "cached-tts"
    def __init__(self, inner, max_items=64):
        self.inner, self.max_items = inner, max_items
        self.cache, self.hits, self.misses = {}, 0, 0
    def synth(self, text):
        if text in self.cache:
            self.hits += 1
            return self.cache[text]
        self.misses += 1
        rec = self.inner.synth(text)
        if len(self.cache) >= self.max_items:
            self.cache.pop(next(iter(self.cache)))       # 단순 FIFO 축출 (과제: LRU)
        self.cache[text] = rec
        return rec

random.seed(2)
totals, stage_ms = profile(VoiceAgentPipeline(MockASR(), MockLLM(), MockTTS()), "utt_001", n=12)
assert pctl(totals, 95) >= pctl(totals, 50)              # 백분위 단조성

r_v2 = VoiceAgentPipelineV2(MockASR_Noisy(), MockLLM(), MockTTS()).run({"pcm": None}, "utt_001")
assert r_v2["reprompt"] is True
assert all(t["stage"] != "LLM" for t in r_v2["traces"])  # 저신뢰엔 LLM 비용을 쓰지 않는다

c = CachedTTS(MockTTS())
c.synth("정합성 검사 문장입니다."); rec = c.synth("정합성 검사 문장입니다.")
assert validate_tts_contract(rec) and c.hits == 1 and c.misses == 1
print(f"최적화 검증 통과 ✅ — p95 {pctl(totals,95):.0f}≥p50 {pctl(totals,50):.0f} / reprompt(LLM 미호출) / 캐시 히트 {c.hits}")


In [ ]:
# ═══ 2.4 직렬 vs 오버랩 + 언더런 검출 (6-2) ✅ ═══
# ▶ 직렬: LLM이 다 끝난 뒤 합성. 오버랩: 문장 도착 즉시 합성(생산-소비 스레드) → TTFA 단축.
#   playback_timeline: 준비 시각+길이로 재생 커튼을 시뮬레이션 → 언더런(끊김) 검출.
import queue, threading

class MockStreamLLM:
    # 문장 단위로 흘려보내는 LLM — 문장당 지연 0.15s (스트리밍 버퍼링 가정)
    name = "mock-stream-llm"
    REPLIES = {"tech_support": ["네, 확인해 드리겠습니다.",
                                "해당 지역 회선을 점검한 결과 신호 불안정이 확인됩니다.",
                                "내일 오전 기사 방문 예약을 도와드리겠습니다."]}
    def stream(self, intent):
        for s in self.REPLIES[intent]:
            time.sleep(0.15)
            yield s

def run_serial(llm_stream, tts):
    # v1 방식: LLM이 전부 끝난 뒤에야 TTS 시작
    t0 = time.perf_counter()
    sents = list(llm_stream.stream("tech_support"))
    recs = [tts.synth(s) for s in sents]
    total = (time.perf_counter() - t0) * 1000
    return {"mode": "직렬", "ttfa_ms": total, "total_ms": total, "n": len(recs)}

def run_overlap(llm_stream, tts):
    # 문장 도착 즉시 합성 — 생성(생산자)과 합성(소비자)이 겹친다
    q = queue.Queue()
    t0 = time.perf_counter()
    def producer():
        for s in llm_stream.stream("tech_support"):
            q.put(s)
        q.put(None)
    threading.Thread(target=producer, daemon=True).start()
    ttfa = None; n = 0
    while True:
        s = q.get()
        if s is None:
            break
        rec = tts.synth(s); n += 1
        if ttfa is None:
            ttfa = (time.perf_counter() - t0) * 1000     # 첫 오디오 준비 완료 시점
    total = (time.perf_counter() - t0) * 1000
    return {"mode": "오버랩", "ttfa_ms": ttfa, "total_ms": total, "n": n}

def playback_timeline(ready_ms, dur_ms):
    # 청크 합성 완료 시각(ready)과 오디오 길이(dur)로 재생 타임라인·언더런(끊김) 계산
    play_start, play_end, gaps = [], [], []
    cursor = 0.0
    for r, d in zip(ready_ms, dur_ms):
        start = max(cursor, r)
        gap = start - cursor if cursor > 0 else 0.0
        if gap > 1.0:
            gaps.append(round(gap, 1))
        play_start.append(start); play_end.append(start + d)
        cursor = start + d
    return play_start, play_end, gaps

random.seed(3)
tts = MockTTS()
r_ser = run_serial(MockStreamLLM(), tts)
r_ovr = run_overlap(MockStreamLLM(), tts)
assert r_ovr["ttfa_ms"] < r_ser["ttfa_ms"]              # 오버랩이 첫 오디오를 더 빨리 준다
assert r_ser["n"] == r_ovr["n"] == 3

# 언더런 검출기 양방향: 정상(커튼 유지)엔 침묵, 위반엔 검출
ps, pe, gaps_ok = playback_timeline([100, 200, 300], [500, 500, 500])
assert gaps_ok == []
ps2, pe2, gaps_bad = playback_timeline([100, 3000], [500, 500])   # 두 번째가 늦음 → 침묵
assert len(gaps_bad) == 1 and gaps_bad[0] > 2000
print(f"오버랩 검증 통과 ✅ — TTFA 직렬 {r_ser['ttfa_ms']:.0f}ms → 오버랩 {r_ovr['ttfa_ms']:.0f}ms / 언더런 양방향")


#### 📄 2.4 보충 — 스트리밍 서버 agent_v3 (6-3·6-4) 시그니처
```python
async def agent_v3(ws: WebSocket):
    await ws.accept()
    utt_id = await ws.receive_text()          # 첫 텍스트 프레임
    pcm = await ws.receive_bytes()            # 오디오 PCM
    asr_rec = await asyncio.to_thread(asr_s.transcribe, audio, utt_id)   # 블로킹은 to_thread
    llm_rec = await asyncio.to_thread(llm_s.generate, asr_rec["text"])
    it = stream_llm_s.stream(llm_rec["intent"])
    seq = 0
    while True:
        sent = await asyncio.to_thread(next, it, None)
        if sent is None: break
        rec = await asyncio.to_thread(tts_s.synth, sent)
        await ws.send_json({"type": "chunk", "seq": seq, "text": sent,
                            "dur_ms": round(rec["duration_s"] * 1000, 1)})
        await ws.send_bytes(pcm16.tobytes())
        seq += 1
    await ws.send_json({"type": "end", "n_chunks": seq})
```
> **교훈**: 블로킹(모델 추론)은 전부 `asyncio.to_thread` — 이벤트 루프는 항상 자유. 프레임 뭉침 방지가 WS의 핵심.
> **바인딩 격리**: 내부 IP → loopback(A) 불가 / 외부(B) 가능 — `smoke_test_64`가 실측.
> **배포 5종**: server.py · requirements.txt · Dockerfile · deploy_cloudrun.sh · deploy_gce_t4.sh.


In [ ]:
# ═══ 2.5 통합 테스트 패턴 — 계약 테스트·장애 주입·playback 회귀 (6-5) ✅ ═══
# ▶ 통합 테스트: 계약 테스트(정상+위반) · 장애 주입(x-fault=llm → fallback) · playback 회귀.
#   '카오스 스위치는 기본 OFF' — 장애 주입이 꺼진 상태가 정상 경로의 기본이다.
def good_tts():
    return make_tts_record(np.zeros(16000, np.float32), 16000, "정상 문장")

def test_tts_ok_passes():
    assert validate_tts_contract(good_tts())

def test_tts_violations_raise(mutate):
    rec = good_tts()
    mutate(rec)
    try:
        validate_tts_contract(rec)
        return False
    except AssertionError:
        return True

def test_asr_ok_passes():
    assert validate_asr_contract(make_asr_record("utt_001", "텍스트", "ko", 0.9, "e", 10.0))

def test_asr_violations_raise(field, value):
    rec = make_asr_record("utt_001", "텍스트", "ko", 0.9, "e", 10.0)
    rec[field] = value
    try:
        validate_asr_contract(rec)
        return False
    except AssertionError:
        return True

def test_wire_ok_and_violation():
    assert validate_wire_meta({"type": "chunk", "seq": 0, "text": "x", "dur_ms": 1.0})
    assert validate_wire_meta({"type": "end"})
    try:
        validate_wire_meta({"type": "chunk", "seq": 0})
        return False
    except AssertionError:
        return True

def pick_llm(fault):
    # 장애 주입: fault=="llm"이면 반드시 실패하는 엔진으로 교체 (카오스 스위치는 기본 OFF)
    return BrokenLLM() if fault == "llm" else MockLLM()

assert test_tts_ok_passes() is None or True
assert test_tts_violations_raise(lambda r: r.__setitem__("sr", 24000))
assert test_tts_violations_raise(lambda r: r.__setitem__("num_channels", 2))
assert test_asr_violations_raise("confidence", 1.5)
assert test_asr_violations_raise("text", "")
assert test_wire_ok_and_violation()

random.seed(4)
# 장애 주입으로 폴백 경로 확인 — fallback 오케스트레이터 + BrokenLLM
r = VoiceAgentPipeline(MockASR(), pick_llm("llm"), MockTTS(), error_policy="fallback").run({"pcm": None}, "utt_002")
assert r["llm"]["handoff_to_human"] is True
r2 = VoiceAgentPipeline(MockASR(), pick_llm(""), MockTTS(), error_policy="fallback").run({"pcm": None}, "utt_002")
assert r2["llm"]["intent"] == "tech_support"             # 기본(카오스 OFF)은 정상 경로

# playback 회귀: 커튼 유지(침묵 없음) + 언더런(끊김) 검출
ps, pe, gaps = playback_timeline([100, 200, 300], [500, 500, 500])
assert gaps == []
ps2, pe2, gaps_bad = playback_timeline([100, 3000], [500, 500])
assert len(gaps_bad) == 1 and gaps_bad[0] > 2000
print("통합 테스트 검증 통과 ✅ — TTS/ASR/wire 계약 테스트 / 장애 주입(fallback vs 정상) / playback 회귀")


In [ ]:
# ═══ 2.6 전송 계약 wire — PCM16 base64 왕복 (6-3) ✅ ═══
# ▶ '계약이 프로세스 경계를 넘는 법': numpy 배열은 JSON을 못 타고, 네트워크는 바이트만 나른다.
#   전송 계약(wire): float32 → int16 PCM → base64 — 과정 표준 16kHz mono PCM_16과 정렬.
#   JSON에 실을 땐 4/3 팽창 세금, WS 바이너리 프레임에선 불필요 (2.7 보충의 근거).
import base64

rec = MockTTS().synth("직렬화 시험 문장입니다.")            # 2.1의 오케스트레이터 자산 그대로 재사용
try:
    json.dumps({"audio": rec["audio"]})
    raise RuntimeError("여기 오면 안 됩니다")
except TypeError as e:
    print("[포착] TypeError:", e)
    print("→ numpy 배열은 JSON을 못 탄다. 프로세스 경계를 넘으려면 '전송 형식'이 따로 필요하다.\n")

def audio_to_wire(audio_f32):
    # float32 → int16 PCM(클립) → base64 문자열 (오디오 표준 PCM_16과 정렬)
    pcm16 = (np.clip(audio_f32, -1.0, 1.0) * 32767).astype(np.int16)
    return base64.b64encode(pcm16.tobytes()).decode("ascii")

def wire_to_audio(b64):
    # base64 → int16 PCM → float32 (1/32767 스케일) — 왕복 복원
    pcm16 = np.frombuffer(base64.b64decode(b64), dtype=np.int16)
    return (pcm16.astype(np.float32) / 32767.0)

wire = audio_to_wire(rec["audio"])
back = wire_to_audio(wire)
assert back.shape == rec["audio"].shape                        # 모양 보존
assert float(np.max(np.abs(back - rec["audio"]))) < 1e-3, "왕복 오차 허용치 초과"   # 값 복원
raw_bytes = rec["audio"].size * 2
print(f"왕복 검증 통과 — PCM {raw_bytes:,}B → base64 {len(wire):,}B (팽창 {len(wire)/raw_bytes:.2f}배)")
print("→ base64는 4/3 팽창을 지불한다. JSON에 실을 땐 편의 비용, WS 바이너리 프레임(2.7 보충)에선 지불 불필요.")


#### 📄 2.6 보충 — 서버 엔드포인트 3종 + WebRTC (6-3) 시그니처

> 서버 기동·WebRTC 실행은 **3-2 가이드** (pip 필요). 여기서는 시그니처로만 요약합니다.
> 핵심: **서버 안에서 도는 것은 6-1의 오케스트레이터 그대로** — 네트워크는 포장일 뿐 내용물은 같다.

**① REST v1 — 한 방 응답 (구조적 한계 실증)**
```python
@app.post("/v1/agent")
async def agent_v1(request: Request):
    body = await request.body()                                  # 원시 PCM16 바이트
    audio = np.frombuffer(body, dtype=np.int16).astype(np.float32) / 32767.0
    asr_rec = asr_s.transcribe(audio, utt_id)                    # 6-1 어댑터 그대로
    llm_rec = llm_s.generate(asr_rec["text"])
    sents = list(stream_llm_s.stream(llm_rec["intent"]))         # 3문장 '전부' 생성될 때까지 대기
    tts_rec = tts_s.synth(" ".join(sents))
    return {"intent": ..., "reply": ..., "audio_b64": audio_to_wire(tts_rec["audio"]), "server_ms": ...}
```
- **한계**: 응답이 한 덩어리 → **TTFA = 총지연** (첫 오디오 = 마지막 바이트). v1은 6-2의 직렬 파이프라인을 네트워크로 옮긴 것에 불과.

**② WS v2 — 전송 프로토콜 계약 (메타 + 바이너리 쌍 + end)**
- 프레임 설계: JSON 메타(`type: chunk|end`, `seq`, `text`, `dur_ms`) + **바이너리 PCM16** 프레임의 쌍. base64 팽창 없음.
- `dur_ms`가 실려 클라이언트가 2.4의 `playback_timeline`(언더런 판정기)을 그대로 돌릴 수 있다. `validate_wire_meta`는 서버·클라이언트 **양쪽**의 같은 게이트.
- **종료 신호 = 계약**: `{"type":"end"}`를 잊으면 ① 서버가 다음 수신을 기다리는 **상호 대기 교착**(클라이언트 `TimeoutError`) 또는 ② close 없이 반환되는 **비정상 종료**(`ConnectionClosedError`) — 어느 쪽이든 클라이언트는 완료를 확정하지 못한다.

**③ WS v3 — 교정판: asyncio의 제1 죄악 해결**
```python
asr_rec = await asyncio.to_thread(asr_s.transcribe, audio, utt_id)   # 블로킹 → 워커 스레드
llm_rec = await asyncio.to_thread(llm_s.generate, asr_rec["text"])
sent = await asyncio.to_thread(next, it, None)                        # 블로킹 제너레이터 소비도 스레드로
```
- **프레임 뭉침 사고**: `async def` 핸들러 안의 `time.sleep`(실물: GPU 추론·외부 API 호출)이 **이벤트 루프를 잠그면**, `await send_bytes()`는 전송 버퍼에 쓸 뿐이고 실제 플러시는 루프가 한가할 때 일어난다 → 핸들러가 끝나는 순간 프레임이 한꺼번에 쏟아진다. TTFA는 REST보다 오히려 나쁘고, **다른 연결도 함께 정지**한다.
- **교훈**: async 핸들러 속 모든 블로킹은 `asyncio.to_thread` — **루프는 항상 자유로워야 한다**.
- **회귀 게이트**: `arrive[-1] - arrive[0] > 500` (뭉침 없음) · `WS_TTFA < REST_TTFA` · 언더런 0.

**④ WebRTC — 왜 WS로 충분하지 않은가 (개념)**
- WS는 TCP — **모든 바이트를 순서대로, 빠짐없이** 배달. 재전송 대기(head-of-line blocking)로 늦은 패킷이 뒤 유효 패킷까지 막는다 (파일엔 미덕, 실시간 음성엔 저주).
- 전화 품질 음성은 **"늦은 패킷은 버리고 간다"** — UDP 계열(SRTP) + 지터 버퍼·AEC(1주차)·대역폭 적응을 표준으로 내장.
- 시그널링(SDP 오퍼/앤서)은 표준 **밖** — 우리 몫, 보통 WS가 배달부가 된다.
- **SDP 해부 3줄**: `m=application ... webrtc-datachannel`(이 연결로 데이터 채널 협상) · `a=ice-ufrag`(NAT 통과 자격 증명) · `a=fingerprint`(DTLS 암호화 신원). 오디오 트랙을 실으면 `m=audio ... OPUS`가 추가되고 16kHz PCM이 48kHz Opus로 재협상된다.
- **데모 요약** (aiortc, 3-2 가이드): 같은 프로세스에 피어 2개를 두고 오퍼/앤서를 **변수로 직접** 교환 → `createDataChannel("aicc-meta")` → pc2가 받은 즉시 에코 → RTT 계측 + 에코된 wire 메타를 `validate_wire_meta`로 검증. (실배포에선 NAT 사이에서 STUN/TURN 필요 — 6-4)

**⑤ 스모크 테스트 `smoke_test_63`** — 살아 있는 서버 대상: 헬스 · REST 응답 키 계약 · wire 왕복 정합 · WS 청크 순서·개수·end · 프레임 뭉침 없음 · TTFA 개선. 6-5의 예고편.


## 2.7 📄 요약 — 카드 3장

### 카드 1: 시스템 통합 계층
| 층 | 구성 | 검증 |
|---|---|---|
| 어댑터 | BaseASR/BaseLLM/BaseTTS + Mock 엔진 | 조합 매트릭스 2×2×1 |
| 오케스트레이터 | VoiceAgentPipeline (예산·오류정책·트레이스) | 예산 판정 · fallback/fail_fast |
| 최적화 | reprompt 게이트 · CachedTTS · 오버랩 | p95·LLM 미호출·TTFA·언더런 |
| 전송 | wire 계약(PCM16) · REST v1 · WS v2/v3 · WebRTC | 왕복 오차·종료 신호·뭉침 없음·TTFA |
| 테스트 | pytest 스위트 · 장애 주입 · smoke_test_63 | 종료코드 판정 · 회귀 |

### 카드 2: 지연 예산과 최적화 축
| 축 | 수단 | 지표 |
|---|---|---|
| 불필요 호출 제거 | reprompt 게이트(conf<0.6) | LLM 호출 0 |
| 반복 비용 제거 | CachedTTS 상용구 | 히트 후 ~0ms |
| 직렬→오버랩 | 문장 도착 즉시 합성 | TTFA 단축 |
| 꼬리 지연 | p95 계측 | p95≥p50 |
| 재생 끊김 | playback_timeline | 언더런 gap>1.0s 검출 |
| 전송 지연 | REST 한 방 → WS 스트리밍 | WS TTFA < REST TTFA |

### 카드 3: 서버 구축의 3계명 (6-3)
1. **전송도 계약이다** — 프레임 형식(PCM16)·순서(seq)·그리고 "끝났다"는 신호(`{"type":"end"}`)까지.
2. **루프는 항상 자유** — async 핸들러 속 모든 블로킹(GPU 추론·API 호출)은 `asyncio.to_thread`로 추방. 블로킹이 루프를 잠그면 프레임이 뭉치고 **다른 연결까지 정지**한다.
3. **실시간 음성은 "늦은 패킷은 버린다"** — TCP(HoL) 대신 WebRTC의 UDP/SRTP + 지터 버퍼가 적시성을 산다.

**교훈 4줄**:
1. 계약은 "위반을 먼저 재현"하고 어댑터로 **엔진 모양을 흡수**한다.
2. 성능 개선은 **계측(p95) 먼저** — 지름길이 아니라 기준선이 먼저다.
3. 오류는 폴백(안전 응답)이든 fail_fast든 **트레이스에 기록**된다 — 조용한 실패가 없다.
4. 테스트의 진실은 출력이 아니라 **종료코드** — `test_` 명명 규약 자체가 게이트다.


## 2.8 [REAL] 실물 실행 — 서버 기동 · REST/WS 계측 · WebRTC (6-3을 로컬로)

> 6-3 모듈을 macOS 로컬에서 그대로 실행합니다: uvicorn 스레드 → REST/WS 클라이언트 → TTFA·프레임 뭉침·
> `to_thread` 교정 실측 → aiortc DataChannel 루프백. 준비: `bash setup_apple_silicon.sh server`
> (셀 실행 후 서버는 종료됩니다 — 포트 반납)


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not ae.has("fastapi", "uvicorn", "httpx"):
    print("서버 스택 미설치 → 스킵.  bash setup_apple_silicon.sh server")
else:
    import threading, asyncio, json, time
    import numpy as np
    from fastapi import FastAPI, Request
    from fastapi.websockets import WebSocket
    import uvicorn
    import httpx

    PORT = 8631
    app = FastAPI(title="AICC Voice Agent API (my-lab 07)")

    # 6-3과 동일: 서버 안에서 도는 것은 이 노트의 오케스트레이터 그대로 — 네트워크는 포장일 뿐
    asr_s, llm_s, tts_s = MockASR(), MockLLM(), MockTTS()

    class _ServerStream:
        # 6-3의 스트림 LLM — 문장당 400ms (v2: 루프를 잠그면 뭉침 / v3: to_thread면 흐름)
        name = "server-stream-llm"
        REPLIES = ["네, 확인해 드리겠습니다.",
                   "해당 지역 회선을 점검한 결과 신호 불안정이 확인됩니다.",
                   "내일 오전 기사 방문 예약을 도와드리겠습니다."]
        def stream(self, intent):
            for s in self.REPLIES:
                time.sleep(0.4)
                yield s
    stream_s = _ServerStream()

    @app.get("/health")
    def health():
        return {"ok": True, "engines": [asr_s.name, llm_s.name, tts_s.name]}

    @app.post("/v1/agent")
    async def agent_v1(request: Request):
        t0 = time.perf_counter()
        body = await request.body()
        utt_id = request.headers.get("x-utt-id", "utt_002")
        audio = np.frombuffer(body, dtype=np.int16).astype(np.float32) / 32767.0
        asr_rec = asr_s.transcribe(audio, utt_id)
        llm_rec = llm_s.generate(build_prompt(asr_rec["text"]))
        sents = list(stream_s.stream("tech_support"))          # 같은 3문장 — WS와 동일 비용
        tts_rec = tts_s.synth(" ".join(sents))
        server_ms = (time.perf_counter() - t0) * 1000
        return {"intent": llm_rec["intent"], "reply": " ".join(sents),
                "sr": tts_rec["sr"], "audio_b64": audio_to_wire(tts_rec["audio"]),
                "server_ms": round(server_ms, 1)}

    @app.websocket("/v2/agent")
    async def agent_v2(ws: WebSocket):
        # 결함판: 블로킹을 to_thread 없이 → 루프 점유 → 프레임 뭉침 재현
        await ws.accept()
        _ = await ws.receive_text(); _ = await ws.receive_bytes()
        for i, sent in enumerate(stream_s.stream("tech_support")):
            rec = tts_s.synth(sent)
            await ws.send_json({"type": "chunk", "seq": i, "text": sent,
                                "dur_ms": round(rec["duration_s"] * 1000, 1)})
            await ws.send_bytes(audio_to_wire(rec["audio"]).encode("ascii"))
        await ws.send_json({"type": "end", "n_chunks": 3})
        await ws.close()

    @app.websocket("/v3/agent")
    async def agent_v3(ws: WebSocket):
        # 교정판: 모든 블로킹을 asyncio.to_thread — 루프는 항상 자유
        await ws.accept()
        _ = await ws.receive_text(); _ = await ws.receive_bytes()
        it = stream_s.stream("tech_support")
        seq = 0
        while True:
            sent = await asyncio.to_thread(next, it, None)
            if sent is None:
                break
            rec = await asyncio.to_thread(tts_s.synth, sent)
            pcm16 = (np.clip(rec["audio"], -1, 1) * 32767).astype(np.int16)
            meta = {"type": "chunk", "seq": seq, "text": sent,
                    "dur_ms": round(rec["duration_s"] * 1000, 1)}
            validate_wire_meta(meta)
            await ws.send_json(meta)
            await ws.send_bytes(pcm16.tobytes())
            seq += 1
        await ws.send_json({"type": "end", "n_chunks": seq})
        await ws.close()

    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(50):
        try:
            r = httpx.get(f"http://127.0.0.1:{PORT}/health", timeout=0.3)
            if r.status_code == 200:
                break
        except Exception:
            time.sleep(0.1)
    else:
        raise RuntimeError("서버 기동 실패 — 포트 충돌이면 재시작 후 재실행")
    print("서버 기동 확인:", r.json())


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if ae.has("fastapi", "uvicorn", "httpx"):
    import json, time
    import numpy as np
    import httpx
    from websockets.sync.client import connect

    fake_pcm = np.zeros(16000, dtype=np.int16).tobytes()

    def run_ws_client(path):
        arrive, durs, texts = [], [], []
        t0 = time.perf_counter()
        with connect(f"ws://127.0.0.1:{PORT}" + path) as ws:
            ws.send("utt_002"); ws.send(fake_pcm)
            pending = None
            while True:
                msg = ws.recv(timeout=10.0)
                if isinstance(msg, str):
                    m = json.loads(msg); validate_wire_meta(m)
                    if m["type"] == "end":
                        break
                    pending = m
                else:
                    arrive.append((time.perf_counter() - t0) * 1000)
                    durs.append(pending["dur_ms"]); texts.append(pending["text"])
        return arrive, durs, texts

    t0 = time.perf_counter()
    resp = httpx.post(f"http://127.0.0.1:{PORT}/v1/agent", content=fake_pcm,
                      headers={"x-utt-id": "utt_002"}, timeout=10)
    REST_TTFA = (time.perf_counter() - t0) * 1000
    data = resp.json()
    print("REST 응답:", data["intent"], f"| server_ms {data['server_ms']} | TTFA(한 방) {REST_TTFA:.0f}ms")

    arrive2, _, _ = run_ws_client("/v2/agent")
    spread2 = arrive2[-1] - arrive2[0] if len(arrive2) > 1 else 0.0
    print(f"WS v2 도착(ms): {[round(a) for a in arrive2]}  → 간격 {spread2:.0f}ms")
    if spread2 < 100:
        print("   ⚠️ [포착] 프레임 뭉침 — 6-3의 결함이 재현됨 (블로킹이 루프를 잠궜다)")
    else:
        print("   이 환경에서는 프레임이 흘러나왔습니다 — 6-3 교훈: '환경 의존적 통과는 결함의 증거'")

    arrive3, durs3, _ = run_ws_client("/v3/agent")
    WS_TTFA = arrive3[0]
    spread3 = arrive3[-1] - arrive3[0] if len(arrive3) > 1 else 0.0
    _, _, gaps = playback_timeline(arrive3, durs3)          # (play_start, play_end, gaps) 3-튜플
    print(f"WS v3 도착(ms): {[round(a) for a in arrive3]}  → 스트리밍 간격 {spread3:.0f}ms")
    print(f"TTFA: REST {REST_TTFA:.0f}ms → WS v3 {WS_TTFA:.0f}ms | 언더런: {gaps if gaps else '없음'}")

    server.should_exit = True                                # 포트 반납 (판정과 무관하게)
    assert spread3 > 500, "v3는 프레임이 시간 간격을 두고 흘러야 (to_thread 교정)"
    assert WS_TTFA < REST_TTFA, "WS 스트리밍 TTFA < REST TTFA"
    assert not gaps, "localhost 언더런 없음"
    print("REST/WS 실측 통과 ✅ — 6-3의 부등식(전송 계약 + to_thread)이 로컬에서 성립")
else:
    print("서버 스택 미설치 → 스킵")


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

# Python 3.14 + aiortc 1.15: wait_for 는 task 안에서만 → 노트북 루프(nest_asyncio)에서 깨짐.
# 별도 프로세스(신선한 루프)로 aicc_webrtc_demo.py 를 실행해 우회한다.
if not ae.has("aiortc"):
    print("aiortc 미설치 → 스킵.  bash setup_apple_silicon.sh server")
else:
    import subprocess
    r = subprocess.run([sys.executable, str(ae.HERE / "aicc_webrtc_demo.py")],
                       capture_output=True, text=True, timeout=60)
    if r.returncode != 0:
        print("WebRTC 실행 실패:", r.stderr[-800:])
    else:
        out = r.stdout
        sdp_part = out.split("###SDP###")[1].split("###RESULT###")[0].strip()
        res = json.loads(out.split("###RESULT###")[1].strip())
        for l in sdp_part.splitlines()[:3]:
            print("  ", l)
        print(f"DataChannel 에코 RTT: {res['rtt_ms']:.1f}ms | wire 메타 검증:", validate_wire_meta(res["echo"]))
        print("WebRTC 루프백 통과 ✅ — UDP/SCTP 제어 채널이 wire 계약을 실어 왕복 (STUN/TURN은 6-4)")


# 3. 실험 진행 방법 🧪

## 3-1. macOS에서 전부 실행 (이 노트 셀 순서)
```
2.0 계약 3종 → 2.1 ABC+오케스트레이터 → 2.2 오류정책+조합 → 2.3 최적화
→ 2.4 직렬/오버랩 → 2.5 통합 테스트 → 2.6 전송 계약 wire 왕복
```
별도 설치·키 없이 위에서 아래로 실행하면 됩니다 (시간 계측이므로 잠깐 기다립니다).

## 3-2. 로컬 서버 기동 (6-3·6-4 — 선택, pip 필요)
```bash
pip install fastapi uvicorn websockets httpx
pip install aiortc nest_asyncio        # WebRTC DataChannel 루프백만 필요
# server.py 를 작성 후 (health + /v1/agent + /v2/agent + /v3/agent)
uvicorn server:app --host 127.0.0.1 --port 8642   # loopback 전용(A)
uvicorn server:app --host 0.0.0.0  --port 8641   # 외부 노출(B)
```
1. **REST 계측**: `httpx.post("/v1/agent", content=fake_pcm)` → 응답 `server_ms`와 클라이언트 총 시간을 분해 (REST는 TTFA=총지연).
2. **WS 계측**: `run_ws_client(utt_id, path="/v2/agent")` → 프레임 뭉침 검출기로 `arrive[-1]-arrive[0]` 관찰 → `/v3/agent` 교정 후 `spread>500` assert가 정식 게이트.
3. **WebRTC 루프백**: `nest_asyncio.apply()` 후 `asyncio.run(datachannel_loopback())` → DataChannel 에코 RTT 계측 + SDP 3줄 해부.
4. **스모크**: `smoke_test_63()` — 살아 있는 서버를 대상으로 전 항목 통과 확인.

## 3-3. pytest 스위트 (6-5 — 선택)
```bash
pip install pytest httpx
mkdir aicc_tests && cd aicc_tests
# aicc_contracts.py / aicc_engines.py / server.py / conftest.py / test_*.py
python -m pytest -q --no-header     # 종료코드가 판정
python -m pytest test_e2e.py -q     # live_server + x-fault 주입
```

## 3-4. GCP 배포 (6-4 — 클라우드, 키 필요)
1. **Cloud Run**: `deploy_cloudrun.sh` — 컨테이너 배포 (서버, 무상태).
2. **GCE T4**: `deploy_gce_t4.sh` — GPU 인스턴스 (모델 상주, 더 큰 VRAM).
3. 배포 전 `smoke_test_64` — 배포 산출물 5종 존재 + 바인딩 격리 + WS 회귀를 **로컬에서 먼저** 통과.

## 3-5. 판단 기준 6종
1. **계약 통과율** — 어댑터가 계약 레코드를 지키는가 (조합 매트릭스 전건).
2. **예산 판정** — 단계별·총합이 BUDGET_MS 안인가 (p50 기준, p95는 여유 대조).
3. **비용 절감** — reprompt로 LLM 호출이 사라졌는가 · 캐시 히트율.
4. **지연/끊김** — TTFA·오버랩 개선 · 언더런 0.
5. **전송 건전성** — wire 왕복 오차 < 1e-3 · `{"type":"end"}` 수신 · 프레임 뭉침 없음(`spread>500`) · WS TTFA < REST TTFA.
6. **회귀 안전** — pytest 종료코드 0 · 장애 주입(fault=llm)에서도 폴백 완주.


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. 4층 계약 아키텍처

```
┌─ 어댑터 층 ──────────────────────────────┐
│  BaseABC ← MockASR/MockLLB/MockTTS (+실물)  │  "엔진 모양 흡수" 벽
├─ 오케스트레이션 층 ────────────────────────┤
│  VoiceAgentPipeline(V2) — 예산·오류·게이트    │  계약 검증 + 트레이스
├─ 서비스 층 ───────────────────────────────┤
│  REST v1(한 방) · WS v2/v3(to_thread)      │  전송 계약: 메타+바이너리+end
│  WebRTC(UDP/SRTP · SDP · DataChannel)      │  실시간 미디어 — HoL 회피
└─ 검증 층 ────────────────────────────────┘
   pytest 스위트 + smoke_test_63 — 계약·파이프라인·서버·E2E (종료코드)
```

## 4-2. 실패의 4가지 소리
| 정책 | 동작 | 기록 |
|---|---|---|
| fail_fast | 예외 전파 (테스트·개발) | 트레이스 없음(중단) |
| fallback | 안전 응답 반환 | `make_trace(..., ok=False, "폴백: ...")` |
| reprompt | 재발화 요청 (LLM 비용 0) | `"저신뢰"` + reprompt |
| isError (wire) | 도구 실패를 결과로 회신 | `{"type":"chunk"..."}` 아님 → MCP/함수호출 연결 |

> **서버의 실패는 한 겹 더 있다**: 종료 신호 부재(교착/비정상 close)와 프레임 뭉침(루프 점유) —
> 둘 다 "응답은 나갔는데 계약이 지켜지지 않은" 경우로, wire 계약 + `to_thread`가 예방한다.

## 4-3. macOS 적용 아키텍처
- **재사용 가능한 로직**(계약·어댑터·게이트·계측·전송 계약)과 **환경 의존**(모델·서버·GCP)을 분리.
- 모델 엔진만 교체하면 동일 오케스트레이터·테스트를 재사용 — "계약이 이식성의 단위".
- 서버 코드(agent_v1/v2/v3)는 로컬에서 그대로 실행 가능 — 6-4에서 별도 프로세스·클라우드로 승격.

## 4-4. 최종 판정
- **6-1**: 부품은 계약으로 꽂고, 위반은 재현으로 검증한다.
- **6-2**: 병목은 p95로 찾고, reprompt·캐시·오버랩 순으로 자른다.
- **6-3**: 전송도 계약이다 — 프레임 형식·순서·종료 신호까지. async 속 블로킹은 to_thread로 추방, 실시간 음성은 "늦은 패킷은 버린다"(WebRTC).
- **6-4**: 서버는 노트북 밖(별도 프로세스·터널·클라우드)으로 — 바인딩·포트·RTT를 실측한다.
- **6-5**: 증명의 단위는 출력이 아니라 종료코드 — 장애 주입을 끄는 것이 기본이다.
